#### Packages

In [ ]:
import skl2onnx
import onnx
import sklearn
from sklearn.linear_model import LogisticRegression
import numpy as np
import onnxruntime as rt
from skl2onnx.common.data_types import FloatTensorType
from skl2onnx import convert_sklearn
from sklearn.ensemble import RandomForestRegressor
import geopandas as gpd
import pandas as pd
import rioxarray as rxr
import xarray as xr
from dask.distributed import Client, LocalCluster
import time
import dask.array as da
import dask
import dask.dataframe as dd
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', message='Sending large graph of size*')

### Load training data and preprocessing

In [ ]:
# Load data
gdf = gpd.read_file(r"original_training_data.gpkg")


# Convert landuse_type into dummies for RF
dum_gdf = pd.get_dummies(gdf, columns = ['landuse_type'])

# use the seed for 25
seed = 25

X = dum_gdf.drop(['soc', 'source', 'geometry'], axis = 1)
y = dum_gdf.loc[:, 'soc']

feature_names = X.columns.tolist()

# Convert X, y into numpy array and make columns into float
X = np.array(X).astype(np.float32)
y = np.array(y).astype(np.float32)

### Load testing data and preprocessing

#### Functions for reading and preprocessing raster data

In [ ]:
def preprocess_raster_data(file_path, band_names, chunk_size=(1000, 1000)):
    """
    Load and preprocess raster data with memory-efficient processing.
    Steps include renaming bands, handling missing data, recoding land use types,
    and filtering out artificial and water areas.
    
    Parameters:
        - file_path (str): Path to the raster file.
        - band_names (list): List of band names.
        - chunk_size (tuple): Size of chunks for lazy processing.
    
    Returns:
        - xarray.Dataset: The preprocessed raster dataset.
        - xarray.DataArray: The original raster data.
    """
    # Landuse recoding dictionary
    landuse_lookup = {
        **{i: 'grassland' for i in range(10100, 12001, 100)},
        **{i: 'wetland' for i in range(20100, 20701, 100)}, 
        **{i: 'forest' for i in range(30100, 31201, 100)},
        **{i: 'arable' for i in range(40100, 40501, 100)},
        **{i: 'coastal' for i in range(50100, 51501, 100)}
    }
    default_value = np.nan  # Default value for codes not in the landuse_lookup
    
    def recode_landuse(da):
        def recode_function(block):
            # Handle floating-point values by rounding and casting to integers
            block = np.rint(block).astype(int)
    
            # Preallocate the recoded array
            recoded = np.empty_like(block, dtype=object)
    
            # Apply the landuse mapping
            for key, value in landuse_lookup.items():
                recoded[block == key] = value
    
            # Assign default value for unmatched keys
            recoded[~np.isin(block, list(landuse_lookup.keys()))] = default_value
    
            return recoded
    
        recoded_data = xr.apply_ufunc(
            recode_function,
            da,
            input_core_dims=[[]],
            output_core_dims=[[]],
            vectorize=False,
            dask="parallelized",
            output_dtypes=[object],
        )
    
        return recoded_data
    
    # Load raster data using rioxarray with larger chunks
    original_raster_data = rxr.open_rasterio(file_path, masked=True, chunks=chunk_size)
    
    # Ensure the number of bands needs to match the band names
    if len(band_names) != original_raster_data.sizes['band']:
        raise ValueError("The number of band names does not match the number of raster bands!")
    
    # Convert the raster data to an xarray Dataset and rename bands
    raster_dataset = original_raster_data.to_dataset(dim='band')
    raster_dataset = raster_dataset.rename({idx + 1: var for idx, var in enumerate(band_names)})
    
    # Apply preprocessing: recoding landuse type
    landuse_band = raster_dataset['landuse_type']
    recoded_landuse_band = recode_landuse(landuse_band)
    raster_dataset['landuse_type'] = recoded_landuse_band
    
    # Replace -9999 with NaN before stacking
    raster_dataset = raster_dataset.where(raster_dataset != -9999, np.nan)
    
    # Process in tiles to avoid memory issues
    y_size, x_size = raster_dataset.y.size, raster_dataset.x.size
    tile_size = 100000 
    
    # Create an empty list to store processed chunks
    processed_chunks = []
    
    # Process in tiles
    for y_start in range(0, y_size, tile_size):
        y_end = min(y_start + tile_size, y_size)
        for x_start in range(0, x_size, tile_size):
            x_end = min(x_start + tile_size, x_size)
            
            # Extract tile
            tile = raster_dataset.isel(y=slice(y_start, y_end), x=slice(x_start, x_end))
            
            # Stack dimensions for this tile
            stacked_tile = tile.stack(dim_0=('y', 'x'))
            
            # Drop NaN values
            filtered_tile = stacked_tile.dropna(dim='dim_0', how='any')
                
            # Store the processed tile
            processed_chunks.append(filtered_tile)
    
    # If there is no valid data; return None
    if not processed_chunks:
        return None, original_raster_data
        
    # Combine all processed tiles into one single dataframe
    final_raster_dataset = xr.concat(processed_chunks, dim= 'dim_0')
    
    return final_raster_dataset, original_raster_data

def convert_to_binary_columns(ds, variable_name, unique_categories):
    """
    Converts a categorical variable in an xarray Dataset into binary columns.
    """
    if variable_name not in ds:
        raise ValueError(f"The variable '{variable_name}' does not exist in the Dataset.")
        
    dummy_ds = xr.Dataset()
    for category in unique_categories:
        binary_variable_name = f"{variable_name}_{category}"
        dummy_ds[binary_variable_name] = xr.where(ds[variable_name] == category, 1, 0)

    ds = ds.drop_vars(variable_name)
    ds = xr.merge([ds, dummy_ds])

    return ds

#### Read data

In [ ]:
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TERMINATE"] = "0.95"  # Allow workers to use up to 95% of memory limit
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__PAUSE"] = "0.85"      # Pause at 85% to prevent crashes
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TARGET"] = "0.8" # Target 80% for optimal performance

cluster = LocalCluster(n_workers=8, threads_per_worker=2, memory_limit='10GB')
client = Client(cluster)
print(f"Dashboard link: {client.dashboard_link}")

# Start the timer
start_time = time.time()

# Define band names for the raster
band_names = ['clay', 'silt', 'sand', 'rock', 'slope', 'lsf', 'tri', 'twi', 'ndvi_spring', 'ndvi_summer', 'ndvi_autumn', 'bsi_spring', 'bsi_summer', 'bsi_autumn', 'drained', 'landuse_type']

# Path to the raster file
file_path = r"estonia_merged_raster.tif"

# Process with fixed function
preprocessed_raster, original_raster = preprocess_raster_data(file_path, band_names)

# Create binary dummy variables for land use type
landuse_cols = sorted(['arable', 'forest', 'grassland', 'wetland'])
dummy_raster_data = convert_to_binary_columns(preprocessed_raster, 'landuse_type', landuse_cols)

# Check the feature order consistency
assert list(dummy_raster_data.data_vars.keys()) == feature_names

# End time
end_time = time.time()

# Calculate elapsed time
execution_time = end_time - start_time
print(f"Execution Time: {execution_time} seconds")

dummy_raster_data

### Compute the prediction with ONNX Runtime

#### Function for prediction

In [ ]:
# Check the avaiablitly of providers
print(rt.get_available_providers())

In [ ]:
def predict_onnx_chunked(raster_dataset, onnx_model_path, chunk_size = 1000000):
    # Load ONNX model
    rf_onnx = rt.InferenceSession(onnx_model_path, providers=['CUDAExecutionProvider'])
    input_name = rf_onnx.get_inputs()[0].name # the inputs metadata
    label_name = rf_onnx.get_outputs()[0].name # the outputs metadata
    
    # Get feature names
    features = list(raster_dataset.data_vars.keys())
    n_samples = raster_dataset.dims['dim_0']
    
    # Process in chunks
    all_predictions = []
    start_time = time.time()
    
    for i in tqdm(range(0, n_samples, chunk_size)):
        end_idx = min(i + chunk_size, n_samples)
        
        # Extract chunk and convert to numpy
        chunk = raster_dataset.isel(dim_0=slice(i, end_idx))
        feature_arrays = [chunk[feature].values for feature in features]
        X_chunk = np.column_stack(feature_arrays).astype(np.float32)
        
        # Predict
        pred_chunk = rf_onnx.run([label_name], {input_name: X_chunk})[0]
        all_predictions.append(pred_chunk)
    
    # Combine predictions
    pred_onx = np.concatenate(all_predictions, axis=0)
    
    # Add predictions to dataset
    preds_1d = pred_onx[:, 0]
    pred_da = xr.DataArray(preds_1d, dims=['dim_0'], coords={'dim_0': raster_dataset['dim_0']}, name='prediction')
    result = raster_dataset.assign(prediction=pred_da)
    
    end_time = time.time()
    print(f"Execution Time: {end_time - start_time:.2f} seconds")
    
    return result

#### Prediction

In [ ]:
# Start the timer
start_time = time.time()

# Prediction
result = predict_onnx_chunked(dummy_raster_data, 'baseline_RF_soc.onnx', chunk_size = 5000000)

# End time
end_time = time.time()

# Calculate elapsed time
execution_time = end_time - start_time
print(f"Execution Time: {execution_time} seconds")

### Save it to the GeoTIFF

In [ ]:
print(result.rio.crs)
print(result.rio.resolution())

In [ ]:
# Define the variables to export
variable_names = ['prediction']

# Get the exact grid definition from the original raster
original_x = original_raster.x.values
original_y = original_raster.y.values
original_transform = original_raster.rio.transform()
original_crs = original_raster.rio.crs

# Create a rasterio transform object once
from rasterio.transform import Affine
inv_transform = ~Affine.from_gdal(*original_transform.to_gdal())

# Define grid shape once
grid_shape = (len(original_y), len(original_x))

# For each variable
for var_name in variable_names:
    print(f"Processing {var_name}...")
    
    # Create the output grid with nodata values
    output_grid = np.full(grid_shape, -9999, dtype=np.float32)
    
    # Get coordinates from result dataset - don't load values yet
    x_coords = result.x
    y_coords = result.y
    
    # Process in larger chunks but not too large to overwhelm memory
    # Adjust chunk_size based on your system's available RAM
    chunk_size = 10000000
    
    # Calculate total number of points
    num_points = len(x_coords.values)
    num_chunks = (num_points + chunk_size - 1) // chunk_size
    
    for i in tqdm(range(num_chunks), desc=f"Processing {var_name} in chunks"):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, num_points)
        
        # Load just this chunk of data
        chunk_slice = slice(start_idx, end_idx)
        
        # Load chunk coordinates
        chunk_x = x_coords.values[chunk_slice]
        chunk_y = y_coords.values[chunk_slice]
        
        # Load chunk values - only compute this chunk if using dask
        if isinstance(result[var_name].data, da.Array):
            chunk_values = result[var_name].data[chunk_slice].compute()
        else:
            chunk_values = result[var_name].values[chunk_slice]
        
        # Vectorized calculation of indices for the entire chunk
        col_indices, row_indices = np.array(inv_transform * (chunk_x, chunk_y)).astype(int)
        
        # Filter valid indices
        valid_mask = (
            (row_indices >= 0) & 
            (row_indices < len(original_y)) & 
            (col_indices >= 0) & 
            (col_indices < len(original_x))
        )
        
        # Apply valid mask to indices and values
        valid_rows = row_indices[valid_mask]
        valid_cols = col_indices[valid_mask]
        valid_values = chunk_values[valid_mask]
        
        # Use np.maximum for overlapping points to keep the highest value
        # Or use another strategy like mean, median, etc. depending on your needs
        for j in range(len(valid_rows)):
            output_grid[valid_rows[j], valid_cols[j]] = valid_values[j]
        
        # Clear memory
        del chunk_x, chunk_y, chunk_values, col_indices, row_indices
        
    # Create DataArray with spatial properties
    output_da = xr.DataArray(
        output_grid,
        dims=["y", "x"],
        coords={"x": original_x, "y": original_y}
    )
    
    # Set spatial properties
    output_da.rio.write_transform(original_transform, inplace=True)
    output_da.rio.write_crs(original_crs, inplace=True)
    output_da.rio.write_nodata(-9999, inplace=True)
    
    # Save to GeoTIFF with memory-efficient settings
    output_da.rio.to_raster(
        f'result/baseline_RF_soc_{var_name}.tif',
        driver="GTiff",
        dtype="float32",
        compress="LZW",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        num_threads="all_cpus"
    )
    
    # Clear memory before processing next variable
    del output_grid, output_da
    import gc
    gc.collect()
    
    print(f"Saved {var_name} to estonia_raster_{var_name}.tif")